In [1]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings("ignore")

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, f1_score, classification_report

# 1. Load and clean datasets
KAGGLE, SYNTH, HARDNEG, GOLD = "cyberbullying_tweets.csv", "comments.csv", "hard_negatives.csv", "gold_to_label.csv"

def clean(t):
    t = str(t).lower()
    t = re.sub(r"http\S+|www\S+", " ", t)
    t = re.sub(r"@\w+", " ", t)
    t = re.sub(r"^rt ", " ", t)
    t = re.sub(r"#", " ", t)
    return " ".join(re.sub(r"[^a-z\s']", " ", t).split())

def load(path):
    d = pd.read_csv(path)[["tweet_text", "cyberbullying_type"]].dropna()
    d = d.drop_duplicates("tweet_text", keep=False)
    d["clean"] = d.tweet_text.map(clean)
    return d.drop_duplicates("clean", keep=False).reset_index(drop=True)

# Define dataset splits
sets = {"Kaggle": load(KAGGLE), "Gemini": load(SYNTH), "HardNeg": load(HARDNEG)}

def split_df(d): 
    return train_test_split(d, test_size=0.2, stratify=d.cyberbullying_type, random_state=42)

parts = {k: split_df(v) if len(v) > 50 else (v, v.iloc[:0]) for k, v in sets.items()}

train_sets = {k: p[0] for k, p in parts.items()}
test_sets = {k: p[1] for k, p in parts.items() if len(p[1])}
test_sets["GOLD"] = load(GOLD)

# Prepare combined training dataframe
tr_combined = pd.concat([train_sets[s].assign(src=s) for s in train_sets])
X_train = tr_combined.clean
y_train = tr_combined.cyberbullying_type

# Sample weights to balance small high-quality sets with massive Kaggle set
sample_weights = np.where(tr_combined.src == "Kaggle", 1.0, 10.0)

# 2. Define Model Configurations for Systematic Experimentation
pipelines = {
    "1. BoW + Logistic Regression": Pipeline([
        ("vect", CountVectorizer(ngram_range=(1, 2), min_df=2)),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))
    ]),
    "2. TF-IDF (Word 1-2) + Naive Bayes": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("clf", MultinomialNB())
    ]),
    "3. TF-IDF (Word 1-2) + Random Forest": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=5, sublinear_tf=True)),
        ("clf", RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42))
    ]),
    "4. TF-IDF (Word 1-2) + Logistic Regression": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", C=5.0))
    ]),
    "5. TF-IDF (Word 1-2) + LinearSVC (Base)": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("clf", CalibratedClassifierCV(LinearSVC(class_weight="balanced", C=1.0, random_state=42)))
    ]),
    "6. TF-IDF (Word 1-2 + Char 3-5) + LinearSVC (Optimized)": Pipeline([
        ("features", FeatureUnion([
            ("word", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
            ("char", TfidfVectorizer(ngram_range=(3, 5), analyzer="char", min_df=3))
        ])),
        ("clf", CalibratedClassifierCV(LinearSVC(class_weight="balanced", C=1.0, random_state=42)))
    ])
}

# 3. Execution & Comparison Loop
results = []

for model_name, pipe in pipelines.items():
    print(f"Training: {model_name}...")
    
    # Train model using sample weights if supported by the classifier
    try:
        pipe.fit(X_train, y_train, clf__sample_weight=sample_weights)
    except TypeError:
        pipe.fit(X_train, y_train)
        
    for tname, te in test_sets.items():
        preds = pipe.predict(te.clean)
        acc = accuracy_score(te.cyberbullying_type, preds)
        macro_f1 = f1_score(te.cyberbullying_type, preds, average="macro")
        
        results.append({
            "Model": model_name,
            "Tested_On": tname,
            "Accuracy": round(acc, 4),
            "Macro_F1": round(macro_f1, 4)
        })

# 4. Display Results Table
df_results = pd.DataFrame(results)
pivot_acc = df_results.pivot(index="Model", columns="Tested_On", values="Accuracy")
pivot_f1 = df_results.pivot(index="Model", columns="Tested_On", values="Macro_F1")

print("\n================ ACCURACY COMPARISON TABLE ================")
print(pivot_acc.to_string())

print("\n================ MACRO F1 COMPARISON TABLE ================")
print(pivot_f1.to_string())

Training: 1. BoW + Logistic Regression...
Training: 2. TF-IDF (Word 1-2) + Naive Bayes...
Training: 3. TF-IDF (Word 1-2) + Random Forest...
Training: 4. TF-IDF (Word 1-2) + Logistic Regression...
Training: 5. TF-IDF (Word 1-2) + LinearSVC (Base)...
Training: 6. TF-IDF (Word 1-2 + Char 3-5) + LinearSVC (Optimized)...

================ ACCURACY COMPARISON TABLE ================
Tested_On                                                 GOLD  Gemini  HardNeg  Kaggle
Model                                                                                  
1. BoW + Logistic Regression                             0.930  0.9139   0.9640  0.8654
2. TF-IDF (Word 1-2) + Naive Bayes                       0.790  0.8944   0.9568  0.7650
3. TF-IDF (Word 1-2) + Random Forest                     0.925  0.8528   0.9496  0.8479
4. TF-IDF (Word 1-2) + Logistic Regression               0.925  0.9472   0.9568  0.8658
5. TF-IDF (Word 1-2) + LinearSVC (Base)                  0.925  0.9389   0.9424  0.8665
6. TF

In [2]:
import pandas as pd
import numpy as np
import re, joblib, warnings
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, confusion_matrix

# 1. Load Data
KAGGLE, SYNTH, HARDNEG, GOLD = "cyberbullying_tweets.csv", "comments.csv", "hard_negatives.csv", "gold_to_label.csv"

def clean(t):
    t = str(t).lower()
    t = re.sub(r"http\S+|www\S+", " ", t)
    t = re.sub(r"@\w+", " ", t)
    t = re.sub(r"^rt ", " ", t)
    t = re.sub(r"#", " ", t)
    return " ".join(re.sub(r"[^a-z\s']", " ", t).split())

def load(path):
    d = pd.read_csv(path)[["tweet_text", "cyberbullying_type"]].dropna()
    d = d.drop_duplicates("tweet_text", keep=False)
    d["clean"] = d.tweet_text.map(clean)
    return d.drop_duplicates("clean", keep=False).reset_index(drop=True)

sets = {"Kaggle": load(KAGGLE), "Gemini": load(SYNTH), "HardNeg": load(HARDNEG)}

def split_df(d): 
    return train_test_split(d, test_size=0.2, stratify=d.cyberbullying_type, random_state=42)

parts = {k: split_df(v) if len(v) > 50 else (v, v.iloc[:0]) for k, v in sets.items()}

train_sets = {k: p[0] for k, p in parts.items()}
test_sets = {k: p[1] for k, p in parts.items() if len(p[1])}
test_sets["GOLD"] = load(GOLD)

tr_combined = pd.concat([train_sets[s].assign(src=s) for s in train_sets])
X_train = tr_combined.clean
y_train = tr_combined.cyberbullying_type
sample_weights = np.where(tr_combined.src == "Kaggle", 1.0, 10.0)

# 2. Define Models
models = {
    "1. BoW + LR": Pipeline([
        ("vect", CountVectorizer(ngram_range=(1, 2), min_df=2)),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))
    ]),
    "2. TFIDF + Naive Bayes": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("clf", MultinomialNB())
    ]),
    "3. TFIDF + Random Forest": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=5, sublinear_tf=True)),
        ("clf", RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42))
    ]),
    "4. TFIDF + LR": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", C=5.0))
    ]),
    "5. TFIDF + LinearSVC (Base)": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("clf", CalibratedClassifierCV(LinearSVC(class_weight="balanced", C=1.0, random_state=42)))
    ]),
    "6. TFIDF (Word+Char) + LinearSVC (Optimized)": Pipeline([
        ("features", FeatureUnion([
            ("word", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
            ("char", TfidfVectorizer(ngram_range=(3, 5), analyzer="char", min_df=3))
        ])),
        ("clf", CalibratedClassifierCV(LinearSVC(class_weight="balanced", C=1.0, random_state=42)))
    ])
}

# 3. Train and Generate Detailed Class-Level Diagnostics
trained_models = {}

for m_name, pipe in models.items():
    print(f"\n=======================================================")
    print(f" TRAINING & EVALUATING DIAGNOSTICS FOR: {m_name}")
    print(f"=======================================================")
    
    try:
        pipe.fit(X_train, y_train, clf__sample_weight=sample_weights)
    except TypeError:
        pipe.fit(X_train, y_train)
        
    trained_models[m_name] = pipe
    
    # Evaluate per test set
    for tname, te in test_sets.items():
        preds = pipe.predict(te.clean)
        print(f"\n--- Model: {m_name} | Test Set: {tname} ---")
        print(classification_report(te.cyberbullying_type, preds, digits=3, zero_division=0))


 TRAINING & EVALUATING DIAGNOSTICS FOR: 1. BoW + LR

--- Model: 1. BoW + LR | Test Set: Kaggle ---
                     precision    recall  f1-score   support

                age      0.968     0.970     0.969      1582
          ethnicity      0.988     0.972     0.980      1551
             gender      0.906     0.843     0.873      1468
  not_cyberbullying      0.650     0.648     0.649      1220
other_cyberbullying      0.647     0.734     0.688      1165
           religion      0.960     0.942     0.951      1588

           accuracy                          0.865      8574
          macro avg      0.853     0.851     0.852      8574
       weighted avg      0.870     0.865     0.867      8574


--- Model: 1. BoW + LR | Test Set: Gemini ---
                     precision    recall  f1-score   support

                age      1.000     0.600     0.750        40
          ethnicity      1.000     0.929     0.963        42
             gender      0.973     0.878     0.923      